In [18]:
import cv2
import torch
import torchvision.transforms as T
from pytorchvideo.models.hub import slowfast_r50
import numpy as np
import requests
import os

# Load the pre-trained SlowFast model (Kinetics-600)
model = slowfast_r50(pretrained=True)
model.eval()

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Load Kinetics-600 labels from DeepMind repository
def load_kinetics_labels():
    url = "https://raw.githubusercontent.com/google-deepmind/kinetics-i3d/master/data/label_map_600.txt"
    response = requests.get(url)
    if response.status_code != 200:
        raise Exception(f"Failed to load labels from {url}, status code: {response.status_code}")
    
    labels = response.text.strip().splitlines()
    return labels

labels = load_kinetics_labels()
print(f"Number of labels loaded: {len(labels)}")  # Should be 600
if len(labels) != 600:
    raise ValueError(f"Expected 600 labels, but got {len(labels)}")

# Video preprocessing function (adjusted for SlowFast)
def preprocess_video(video_path, num_frames=32, frame_size=(224, 224)):
    if not os.path.exists(video_path):
        raise FileNotFoundError(f"Video file not found at: {video_path}")
    
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError(f"Failed to open video file: {video_path}")
    
    frames = []
    while len(frames) < num_frames and cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = cv2.resize(frame, frame_size)
        frames.append(frame)
    
    cap.release()
    
    if not frames:
        raise ValueError(f"No frames could be read from video: {video_path}")
    
    while len(frames) < num_frames:
        frames.append(frames[-1])
    
    # Convert to tensor and permute to (T, C, H, W)
    video_clip = np.stack(frames, axis=0)  # Shape: (T, H, W, C)
    video_clip = torch.tensor(video_clip).permute(0, 3, 1, 2).float() / 255.0  # Shape: (T, C, H, W)
    
    # Normalize each frame individually
    transform = T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    video_clip = torch.stack([transform(frame) for frame in video_clip])  # Shape: (T, C, H, W)
    
    # Prepare slow and fast pathways
    video_clip = video_clip.permute(1, 0, 2, 3)  # Shape: (C, T, H, W)
    slow_clip = video_clip[:, ::4, :, :]  # Subsample every 4th frame (8 frames)
    fast_clip = video_clip  # Full 32 frames
    return [slow_clip.unsqueeze(0), fast_clip.unsqueeze(0)]  # Shape: [(1, C, 8, H, W), (1, C, 32, H, W)]

# Inference function
def detect_activity(video_clip, model, labels):
    video_clip = [clip.to(device) for clip in video_clip]  # Move both tensors to device
    with torch.no_grad():
        output = model(video_clip)
        probabilities = torch.softmax(output, dim=1)
        top5_prob, top5_class = probabilities.topk(5, dim=1)
        predicted_index = top5_class[0, 0].item()
        predicted_label = labels[predicted_index]
        confidence = top5_prob[0, 0].item()
        
        print("Top 5 predicted activities:")
        for i in range(5):
            idx = top5_class[0, i].item()
            prob = top5_prob[0, i].item()
            print(f"{i+1}. {labels[idx]}: {prob:.4f}")
    
    return predicted_label, confidence

# Main execution
video_path = "H:/MovieDesc/Extracting Frames/detected_frames/cluster_9_video_1033144046/cluster_9_video_1033144046.mp4"  # Replace with your drone video path
try:
    video_clip = preprocess_video(video_path, num_frames=32, frame_size=(224, 224))
    activity, confidence = detect_activity(video_clip, model, labels)

    print(f"Detected Activity: {activity}")
    print(f"Confidence: {confidence:.4f}")

    # Optional: Display video with activity label
    cap = cv2.VideoCapture(video_path)
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        cv2.putText(frame, f"Activity: {activity} ({confidence:.2f})", 
                    (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        cv2.imshow("Video with Activity Detection", frame)
        if cv2.waitKey(30) & 0xFF == ord('q'):
            break
    cap.release()
    cv2.destroyAllWindows()

except Exception as e:
    print(f"Error: {e}")

Number of labels loaded: 600
Top 5 predicted activities:
1. abseiling: 0.5451
2. clay pottery making: 0.4149
3. exercising with an exercise ball: 0.0337
4. eating doughnuts: 0.0038
5. playing pan pipes: 0.0011
Detected Activity: abseiling
Confidence: 0.5451


KeyboardInterrupt: 

In [ ]:
import cv2
import torch
from transformers import CLIPProcessor, CLIPModel
import numpy as np
from PIL import Image

# Load CLIP model and processor
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Define custom text prompts for classification
text_prompts = [
    "waterfall",
    "drone shot of a city",
    "fish swimming",
    "abseiling",
    "clay pottery making"
]

# Video preprocessing function
def preprocess_video(video_path, frame_size=(224, 224)):
    if not os.path.exists(video_path):
        raise FileNotFoundError(f"Video file not found at: {video_path}")
    
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError(f"Failed to open video file: {video_path}")
    
    frames = []
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = cv2.resize(frame, frame_size)
        frames.append(frame)
    
    cap.release()
    
    if not frames:
        raise ValueError(f"No frames could be read from video: {video_path}")
    
    return frames

# Classify video using CLIP
def classify_video(video_path, text_prompts, model, processor, device):
    frames = preprocess_video(video_path)
    
    # Prepare text inputs
    inputs = processor(text=text_prompts, return_tensors="pt", padding=True, truncation=True)
    text_features = model.get_text_features(**inputs.to(device))
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)
    
    # Process frames and aggregate scores
    frame_scores = []
    for frame in frames[:32]:  # Limit to 32 frames for efficiency (adjust as needed)
        # Convert frame to PIL Image
        pil_image = Image.fromarray(frame)
        
        # Preprocess image
        inputs = processor(images=pil_image, return_tensors="pt", padding=True, truncation=True)
        image_features = model.get_image_features(**inputs.to(device))
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)
        
        # Compute similarity (cosine similarity)
        similarity = (image_features @ text_features.T).cpu().detach().numpy()
        frame_scores.append(similarity)
    
    # Debug: Print shapes to understand structure
    print(f"Frame scores shape: {np.array(frame_scores).shape}")
    
    # Average scores across frames
    avg_scores = np.mean(frame_scores, axis=0)
    print(f"Average scores shape: {avg_scores.shape}")
    
    # Ensure avg_scores is 1D
    if avg_scores.ndim > 1:
        avg_scores = avg_scores.flatten()
    
    # Get best prediction
    best_idx = np.argmax(avg_scores)
    confidence = float(avg_scores[best_idx])  # Convert to Python float
    prediction = text_prompts[best_idx]
    
    print("Top predictions (averaged over frames):")
    for i, prompt in enumerate(text_prompts):
        print(f"{i+1}. {prompt}: {float(avg_scores[i]):.4f}")  # Convert to Python float for formatting
    
    return prediction, confidence

# Main execution
video_path = "H:/MovieDesc/Extracting Frames/detected_frames/cluster_9_video_1033144046/cluster_9_video_1033144046.mp4"
try:
    activity, confidence = classify_video(video_path, text_prompts, model, processor, device)
    print(f"Detected Activity: {activity}")
    print(f"Confidence: {confidence:.4f}")

    # Optional: Display video with activity label
    cap = cv2.VideoCapture(video_path)
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        cv2.putText(frame, f"Activity: {activity} ({confidence:.2f})", 
                    (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        cv2.imshow("Video with Activity Detection", frame)
        if cv2.waitKey(30) & 0xFF == ord('q'):
            break
    cap.release()
    cv2.destroyAllWindows()

except Exception as e:
    print(f"Error: {e}")

Frame scores shape: (32, 1, 5)
Average scores shape: (1, 5)
Top predictions (averaged over frames):
1. waterfall: 0.2901
2. drone shot of a city: 0.1702
3. fish swimming: 0.2315
4. abseiling: 0.2442
5. clay pottery making: 0.1639
Detected Activity: waterfall
Confidence: 0.2901


KeyboardInterrupt: 

: 